# 🚉 Railway Platform Safety Monitoring System
### *AI-Based Computer Vision & Safety Line Violation Detection*

This Jupyter Notebook provides a complete interactive walkthrough of the Railway Platform Safety Monitoring System. It allows you to:
1. Generate a synthetic railway platform CCTV video.
2. Run the safety monitoring computer vision pipeline (using YOLOv8 or classical Fallback tracker).
3. Track passengers and detect safety line crossings.
4. Store evidence snapshots of violations.
5. View historical event databases and analyze safety metrics directly within the notebook.

---
## 🛠️ Step 1: Install Dependencies
Run the cell below to verify and install all required python packages.

In [ ]:
# Install dependencies if not already present
# !pip install ultralytics opencv-python pandas matplotlib pillow

---
## ⚙️ Step 2: System Configuration
We define our parameters, directories, safety line coordinates, and thresholds directly below.

In [ ]:
import os

# Directories & File Paths
BASE_DIR = os.getcwd()
EVIDENCE_DIR = os.path.join(BASE_DIR, "evidence")
EVENT_LOG_PATH = os.path.join(BASE_DIR, "event_log.csv")
VIDEO_INPUT = os.path.join(BASE_DIR, "railway_platform_test.mp4")
VIDEO_OUTPUT = os.path.join(BASE_DIR, "output_video.mp4")

os.makedirs(EVIDENCE_DIR, exist_ok=True)

# CV Parameters
YOLO_MODEL_NAME = "yolov8n.pt"
CONFIDENCE_THRESHOLD = 0.4
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

# Default Safety Line: Vertical boundary at x = 750 (Platform is left, Track is right)
DEFAULT_LINE_COORDS = ((750, 0), (750, 720))
SAFE_SIDE = "left"

# Alert Rules
ALERT_COOLDOWN_SECONDS = 3.0
DWELL_TIME_THRESHOLD_SECONDS = 5.0

---
## 📹 Step 3: Generate Synthetic Railway Platform CCTV Video
To test the system, we generate a synthetic platform video where simulated passenger circles/rectangles move around, with some crossing the yellow safety line.

In [ ]:
import cv2
import numpy as np

def create_background():
    bg = np.zeros((FRAME_HEIGHT, FRAME_WIDTH, 3), dtype=np.uint8)
    bg[:, :750] = [180, 180, 180]  # Platform Area (Gray)
    
    # Grid pattern on platform
    for x in range(100, 750, 100):
        cv2.line(bg, (x, 0), (x, FRAME_HEIGHT), (160, 160, 160), 1)
    for y in range(100, FRAME_HEIGHT, 100):
        cv2.line(bg, (0, y), (750, y), (160, 160, 160), 1)
        
    bg[:, 750:] = [45, 50, 55]  # Track Zone (Dark Gray)
    
    # Railroad ties & steel tracks
    for y in range(20, FRAME_HEIGHT, 40):
        cv2.rectangle(bg, (800, y), (1200, y + 15), (30, 40, 50), -1)
    cv2.line(bg, (880, 0), (880, FRAME_HEIGHT), (120, 120, 130), 8)
    cv2.line(bg, (1120, 0), (1120, FRAME_HEIGHT), (120, 120, 130), 8)
    
    # Yellow Safety line
    cv2.line(bg, (750, 0), (750, FRAME_HEIGHT), (0, 215, 255), 10)
    
    # Labels
    cv2.putText(bg, "PASSENGER PLATFORM", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (80, 80, 80), 2)
    cv2.putText(bg, "TRACK ZONE (DANGER)", (800, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 200), 2)
    return bg

def draw_passenger(img, center, color, label):
    cx, cy = center
    cv2.ellipse(img, (cx, cy + 20), (20, 30), 0, 0, 360, color, -1)
    cv2.circle(img, (cx, cy - 20), 15, (220, 200, 180), -1)
    cv2.circle(img, (cx, cy - 25), 15, color, -1)
    cv2.putText(img, label, (cx - 25, cy - 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
    cv2.putText(img, label, (cx - 25, cy - 45), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

def generate_test_video(output_path, num_frames=300):
    bg_base = create_background()
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, 30.0, (FRAME_WIDTH, FRAME_HEIGHT))
    
    for frame_idx in range(num_frames):
        img = bg_base.copy()
        
        # Passenger 1 (Safe - walks on platform)
        p1_x = int(300 + 150 * np.sin(2 * np.pi * frame_idx / 150))
        p1_y = int(250 + 50 * np.cos(2 * np.pi * frame_idx / 150))
        draw_passenger(img, (p1_x, p1_y), (200, 50, 50), "P1 (Safe)")
        
        # Passenger 2 (Crosser - crosses yellow line)
        if frame_idx < 90:
            alpha = frame_idx / 90.0
            p2_x = int(300 + alpha * (900 - 300))
            p2_y = int(450 + alpha * (480 - 450))
        elif frame_idx < 210:
            p2_x = int(900 + 10 * np.sin(2 * np.pi * frame_idx / 30))
            p2_y = int(480 + 5 * np.cos(2 * np.pi * frame_idx / 30))
        else:
            alpha = (frame_idx - 210) / (num_frames - 210)
            p2_x = int(900 - alpha * (900 - 400))
            p2_y = int(480 - alpha * (480 - 420))
        draw_passenger(img, (p2_x, p2_y), (50, 180, 50), "P2 (Crosser)")
        
        # Passenger 3 (Edge - stands close to line)
        p3_x = int(710 + 15 * np.sin(2 * np.pi * frame_idx / 80))
        p3_y = int(180 + 10 * np.cos(2 * np.pi * frame_idx / 80))
        draw_passenger(img, (p3_x, p3_y), (50, 50, 200), "P3 (Edge)")
        
        # Passenger 4 (Fast Crosser - crosses briefly)
        if 100 <= frame_idx < 250:
            t = frame_idx - 100
            if t < 50:
                p4_x = int(500 + (t / 50.0) * 350)
            elif t < 100:
                p4_x = 850
            else:
                p4_x = int(850 - ((t - 100) / 50.0) * 350)
            p4_y = 600
            draw_passenger(img, (p4_x, p4_y), (150, 50, 150), "P4 (Fast)")
            
        out.write(img)
        
    out.release()
    print(f"Synthetic test video created at: {output_path}")

# Run generator
generate_test_video(VIDEO_INPUT)

---
## 🧠 Step 4: Core Safety Monitoring & Tracking Logic
We declare our safety tracking systems. To avoid loading issues in PyTorch 2.6, we monkeypatch `torch.load` to bypass `weights_only` defaults.

In [ ]:
import pandas as pd
import datetime
import time
import threading
import winsound

# Try to import Ultralytics YOLO with PyTorch 2.6 weights_only patch
YOLO_AVAILABLE = False
try:
    import torch
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = _patched_load
    
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
    print("YOLOv8 successfully imported and patched!")
except ImportError:
    print("YOLOv8 not available. The system will auto-fallback to OpenCV Centroid tracker.")

# Asynchronous auditory alert beeper
def play_sound():
    try:
        winsound.Beep(1000, 300)
    except Exception:
        pass

def trigger_beep():
    threading.Thread(target=play_sound, daemon=True).start()


class CentroidTracker:
    def __init__(self, max_disappeared=15):
        self.next_object_id = 1
        self.objects = {}
        self.bboxes = {}
        self.disappeared = {}
        self.max_disappeared = max_disappeared

    def register(self, centroid, bbox):
        self.objects[self.next_object_id] = centroid
        self.bboxes[self.next_object_id] = bbox
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1

    def deregister(self, object_id):
        del self.objects[object_id]
        del self.bboxes[object_id]
        del self.disappeared[object_id]

    def update(self, rects):
        if len(rects) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.bboxes

        input_centroids = np.zeros((len(rects), 2), dtype="int")
        for (i, (x, y, w, h)) in enumerate(rects):
            input_centroids[i] = (int(x + w / 2.0), int(y + h / 2.0))

        if len(self.objects) == 0:
            for i in range(0, len(input_centroids)):
                self.register(input_centroids[i], rects[i])
        else:
            object_ids = list(self.objects.keys())
            object_centroids = list(self.objects.values())

            D = np.linalg.norm(np.array(object_centroids)[:, np.newaxis] - input_centroids, axis=2)
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]

            used_rows = set()
            used_cols = set()

            for (row, col) in zip(rows, cols):
                if row in used_rows or col in used_cols:
                    continue
                object_id = object_ids[row]
                self.objects[object_id] = input_centroids[col]
                self.bboxes[object_id] = rects[col]
                self.disappeared[object_id] = 0
                used_rows.add(row)
                used_cols.add(col)

            unused_rows = set(range(0, D.shape[0])).difference(used_rows)
            for row in unused_rows:
                object_id = object_ids[row]
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            unused_cols = set(range(0, D.shape[1])).difference(used_cols)
            for col in unused_cols:
                self.register(input_centroids[col], rects[col])

        return self.bboxes


def create_static_bg():
    bg = np.zeros((FRAME_HEIGHT, FRAME_WIDTH, 3), dtype=np.uint8)
    bg[:, :750] = [180, 180, 180]
    for x in range(100, 750, 100):
        cv2.line(bg, (x, 0), (x, FRAME_HEIGHT), (160, 160, 160), 1)
    for y in range(100, FRAME_HEIGHT, 100):
        cv2.line(bg, (0, y), (750, y), (160, 160, 160), 1)
    bg[:, 750:] = [45, 50, 55]
    for y in range(20, FRAME_HEIGHT, 40):
        cv2.rectangle(bg, (800, y), (1200, y + 15), (30, 40, 50), -1)
    cv2.line(bg, (880, 0), (880, FRAME_HEIGHT), (120, 120, 130), 8)
    cv2.line(bg, (1120, 0), (1120, FRAME_HEIGHT), (120, 120, 130), 8)
    cv2.line(bg, (750, 0), (750, FRAME_HEIGHT), (0, 215, 255), 10)
    return bg


class SafetyMonitor:
    def __init__(self, use_yolo=True, is_synthetic=False):
        self.use_yolo = use_yolo and YOLO_AVAILABLE
        self.is_synthetic = is_synthetic
        
        if self.use_yolo:
            print(f"Loading YOLO model: {YOLO_MODEL_NAME}...")
            try:
                self.model = YOLO(YOLO_MODEL_NAME)
            except Exception as e:
                print(f"YOLO load failed: {e}. Falling back to OpenCV tracking.")
                self.use_yolo = False
                
        if not self.use_yolo:
            if self.is_synthetic:
                print("Initializing Fallback Static Background Diff Safety Monitor...")
                self.static_bg = create_static_bg()
            else:
                print("Initializing Fallback MOG2 Background Subtraction tracker...")
                self.back_sub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=30, detectShadows=True)
            self.tracker = CentroidTracker()
            
        self.track_states = {}
        self.event_records = []
        
        # Initialize empty log CSV
        pd.DataFrame(columns=["Timestamp", "Track ID", "Event Type", "Duration in Danger Zone (s)", "Snapshot Path"]).to_csv(EVENT_LOG_PATH, index=False)

    def is_point_in_danger(self, point, line_coords, safe_side):
        px, py = point
        (x1, y1), (x2, y2) = line_coords
        is_vertical = abs(x2 - x1) < abs(y2 - y1)
        
        if is_vertical:
            x_line = x1 + (py - y1) * (x2 - x1) / (y2 - y1) if y2 != y1 else x1
            return px > x_line if safe_side == "left" else px < x_line
        else:
            y_line = y1 + (px - x1) * (y2 - y1) / (x2 - x1) if x2 != x1 else y1
            return py > y_line if safe_side == "top" else py < y_line

    def log_event(self, track_id, event_type, duration, snapshot_path):
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        record = {
            "Timestamp": timestamp,
            "Track ID": track_id,
            "Event Type": event_type,
            "Duration in Danger Zone (s)": round(duration, 2),
            "Snapshot Path": snapshot_path
        }
        self.event_records.append(record)
        pd.DataFrame(self.event_records).to_csv(EVENT_LOG_PATH, index=False)
        print(f"[{timestamp}] ALERT: ID {track_id} - {event_type} ({duration:.1f}s)")

    def save_evidence_snapshot(self, frame, track_id, event_type, bbox):
        x, y, w, h = bbox
        annotated = frame.copy()
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 0, 255), 2)
        cv2.putText(annotated, f"VIOLATION ID: {track_id}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        cv2.line(annotated, DEFAULT_LINE_COORDS[0], DEFAULT_LINE_COORDS[1], (0, 0, 255), 3)
        
        filename = f"track_{track_id}_{event_type.lower().replace(' ', '_')}_{int(time.time())}.jpg"
        filepath = os.path.join(EVIDENCE_DIR, filename)
        cv2.imwrite(filepath, annotated)
        return filepath

    def process_frame(self, frame, line_coords=DEFAULT_LINE_COORDS, safe_side=SAFE_SIDE):
        h_frame, w_frame, _ = frame.shape
        if (w_frame, h_frame) != (FRAME_WIDTH, FRAME_HEIGHT):
            frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
            
        annotated_frame = frame.copy()
        current_time = time.time()
        detected_people = []
        
        if self.use_yolo:
            results = self.model.track(frame, persist=True, verbose=False, conf=CONFIDENCE_THRESHOLD)
            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes
                for i in range(len(boxes)):
                    if int(boxes.cls[i]) == 0:
                        x1, y1, x2, y2 = map(int, boxes.xyxy[i].cpu().numpy().flatten())
                        track_id = int(boxes.id[i].item()) if boxes.id is not None else i + 1000
                        detected_people.append((track_id, x1, y1, x2 - x1, y2 - y1))
        else:
            if self.is_synthetic:
                diff = cv2.absdiff(frame, self.static_bg)
                fg_mask = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
                _, fg_mask = cv2.threshold(fg_mask, 15, 255, cv2.THRESH_BINARY)
            else:
                fg_mask = self.back_sub.apply(frame)
                _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)
            
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
            fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel)
            contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            rects = []
            for contour in contours:
                area_thresh = 500 if self.is_synthetic else 1500
                if cv2.contourArea(contour) > area_thresh:
                    x, y, w, h = cv2.boundingRect(contour)
                    if h > 25:
                        rects.append((x, y, w, h))
            tracked = self.tracker.update(rects)
            for track_id, (x, y, w, h) in tracked.items():
                detected_people.append((track_id, x, y, w, h))

        any_violation = False
        for track_id, x, y, w, h in detected_people:
            ref_point = (int(x + w / 2.0), y + h)
            in_danger = self.is_point_in_danger(ref_point, line_coords, safe_side)
            
            if track_id not in self.track_states:
                self.track_states[track_id] = {
                    "danger_entry_time": None, "total_danger_duration": 0.0, "in_danger": False,
                    "last_alert_time": 0.0, "violation_logged": False, "dwell_warning_logged": False
                }
                
            state = self.track_states[track_id]
            prev_in_danger = state["in_danger"]
            state["in_danger"] = in_danger
            
            box_color = (0, 255, 0)
            status_text = f"ID:{track_id} SAFE"
            
            if in_danger:
                any_violation = True
                box_color = (0, 0, 255)
                if not prev_in_danger or state["danger_entry_time"] is None:
                    state["danger_entry_time"] = current_time
                    
                state["total_danger_duration"] = current_time - state["danger_entry_time"]
                
                if not state["violation_logged"]:
                    state["violation_logged"] = True
                    snap = self.save_evidence_snapshot(frame, track_id, "Line Crossing", (x, y, w, h))
                    self.log_event(track_id, "Line Crossing", state["total_danger_duration"], snap)
                    trigger_beep()
                    state["last_alert_time"] = current_time
                    
                if state["total_danger_duration"] >= DWELL_TIME_THRESHOLD_SECONDS and not state["dwell_warning_logged"]:
                    state["dwell_warning_logged"] = True
                    snap = self.save_evidence_snapshot(frame, track_id, "Dwell Danger", (x, y, w, h))
                    self.log_event(track_id, "Dwell Danger", state["total_danger_duration"], snap)
                    trigger_beep()
                    state["last_alert_time"] = current_time
                    
                if current_time - state["last_alert_time"] > ALERT_COOLDOWN_SECONDS:
                    trigger_beep()
                    state["last_alert_time"] = current_time
                    
                status_text = f"ID:{track_id} DANGER ({state['total_danger_duration']:.1f}s)"
            else:
                if prev_in_danger and state["danger_entry_time"] is not None:
                    total_dur = current_time - state["danger_entry_time"]
                    snap = self.save_evidence_snapshot(frame, track_id, "Safe Return", (x, y, w, h))
                    self.log_event(track_id, "Safe Return", total_dur, snap)
                    state["danger_entry_time"] = None
                    state["violation_logged"] = False
                    state["dwell_warning_logged"] = False
                    
            cv2.rectangle(annotated_frame, (x, y), (x + w, y + h), box_color, 2)
            cv2.circle(annotated_frame, ref_point, 5, (255, 0, 0), -1)
            cv2.putText(annotated_frame, status_text, (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)
            
        line_color = (0, 0, 255) if any_violation else (0, 215, 255)
        cv2.line(annotated_frame, line_coords[0], line_coords[1], line_color, 3)
        
        if any_violation:
            overlay = annotated_frame.copy()
            cv2.rectangle(overlay, (0, 0), (FRAME_WIDTH, 60), (0, 0, 255), -1)
            cv2.addWeighted(overlay, 0.3, annotated_frame, 0.7, 0, annotated_frame)
            cv2.putText(annotated_frame, "!!! SAFETY VIOLATION DETECTED !!!", (350, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
            
        return annotated_frame

---
## 🎥 Step 5: Run Video Safety Pipeline & Show Live Output
Below, we run our monitor. We display the processed frames directly inside the notebook in real-time by converting them to images.

In [ ]:
from IPython.display import display, clear_output
import PIL.Image
import io

# Choose tracker type: set use_yolo=True to use YOLOv8 or use_yolo=False to use fallback
# Note: is_synthetic=True enables the high-precision background differencing specifically for our simulated shapes
monitor = SafetyMonitor(use_yolo=False, is_synthetic=True) 

cap = cv2.VideoCapture(VIDEO_INPUT)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, 30.0, (FRAME_WIDTH, FRAME_HEIGHT))

frame_count = 0
try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        processed = monitor.process_frame(frame)
        out.write(processed)
        
        # Downsample visual updates in notebook to keep it responsive (every 5th frame)
        if frame_count % 5 == 0:
            rgb_frame = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
            img = PIL.Image.fromarray(rgb_frame)
            
            clear_output(wait=True)
            print(f"Processing frames... Current Frame: {frame_count}/300")
            display(img)
            
        frame_count += 1
        time.sleep(0.01) # Small delay to align rendering
        
except KeyboardInterrupt:
    print("Processing interrupted by user.")
finally:
    cap.release()
    out.release()
    print(f"Finished processing! Annotated output video saved to: {VIDEO_OUTPUT}")

---
## 🎥 Step 5b: Run Safety Pipeline on Custom Real Videos (op1 & op2)
Below, we test the safety monitor on real video feeds downloaded from the internet:
1. **subway.mp4** -> A real subway station platform video. Output saved to **op1.mp4**.
2. **people-walking.mp4** -> A real public concourse passenger traffic video. Output saved to **op2.mp4**.

In [ ]:
# Process Subway Video (op1.mp4)
print("Processing Subway Platform Video (op1.mp4)...")
monitor = SafetyMonitor(use_yolo=True, is_synthetic=False)

cap = cv2.VideoCapture("subway.mp4")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("op1.mp4", fourcc, 30.0, (FRAME_WIDTH, FRAME_HEIGHT))

# Subway platform safety line coordinates (vertical line dividing platform/tracks)
subway_line_coords = ((580, 0), (580, 720))
subway_safe_side = "right"

frame_count = 0
try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        processed = monitor.process_frame(frame, line_coords=subway_line_coords, safe_side=subway_safe_side)
        out.write(processed)
        
        if frame_count % 100 == 0:
            print(f"Processed frame: {frame_count}/1298")
            
        frame_count += 1
except KeyboardInterrupt:
    print("Processing interrupted by user.")
finally:
    cap.release()
    out.release()
    print("Finished processing Subway Video! Saved to op1.mp4")

In [ ]:
# Process People Walking Video (op2.mp4)
print("Processing People Walking Video (op2.mp4)...")
monitor = SafetyMonitor(use_yolo=True, is_synthetic=False)

cap = cv2.VideoCapture("people-walking.mp4")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("op2.mp4", fourcc, 30.0, (FRAME_WIDTH, FRAME_HEIGHT))

# People walking safety line coordinates (vertical line dividing the frame)
walking_line_coords = ((640, 0), (640, 720))
walking_safe_side = "left"

frame_count = 0
try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        processed = monitor.process_frame(frame, line_coords=walking_line_coords, safe_side=walking_safe_side)
        out.write(processed)
        
        if frame_count % 50 == 0:
            print(f"Processed frame: {frame_count}/341")
            
        frame_count += 1
except KeyboardInterrupt:
    print("Processing interrupted by user.")
finally:
    cap.release()
    out.release()
    print("Finished processing People Walking Video! Saved to op2.mp4")

---
## 📊 Step 6: Safety Violation Analytics
We load the event CSV generated during monitoring, display detailed records, and visualize statistics using `matplotlib`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load log file
if os.path.exists(EVENT_LOG_PATH):
    df_events = pd.read_csv(EVENT_LOG_PATH)
    if not df_events.empty():
        print("### Safety Event Logs Table ###")
        display(df_events)
        
        # Plot Event Types distribution
        plt.figure(figsize=(10, 4))
        
        plt.subplot(1, 2, 1)
        df_events["Event Type"].value_counts().plot(kind="bar", color=['#ef4444', '#f59e0b', '#10b981'])
        plt.title("Event Type Frequency")
        plt.ylabel("Count")
        plt.xticks(rotation=45)
        
        plt.subplot(1, 2, 2)
        crossings = df_events[df_events["Event Type"] == "Line Crossing"]
        if not crossings.empty:
            crossings["Track ID"].value_counts().plot(kind="bar", color='purple')
            plt.title("Crossings by Passenger ID")
            plt.xlabel("Passenger ID")
            plt.ylabel("Crossing Count")
        
        plt.tight_layout()
        plt.show()
    else:
        print("Event log is empty. No events logged.")
else:
    print("event_log.csv not found.")

---
## 🖥️ Step 7: Launching the Web Operator Dashboard
To launch the interactive dashboard (Streamlit app) in your browser, open your terminal/command prompt, navigate to this project folder, and run:

```bash
streamlit run app.py
```